Carga Inicial de los Dataset Proprocionados

In [2]:
import numpy as np
import pandas as pd
import streamlit as st

df_customers_all = pd.read_csv('../../documents/customers_dataset.csv') # Correcto
df_order_items_all = pd.read_csv('../../documents/order_items_dataset.csv') # Correcto
df_order_payments_all = pd.read_csv('../../documents/order_payments_dataset.csv') # Correcto
df_order_reviews_all = pd.read_csv('../../documents/order_reviews_dataset.csv') # Correcto
df_orders_all = pd.read_csv('../../documents/orders_dataset.csv') # Correcto
df_product_category_name_translation_all = pd.read_csv('../../documents/product_category_name_translation.csv') # Correcto
df_products_all = pd.read_csv('../../documents/products_dataset.csv')
df_sellers_all = pd.read_csv('../../documents/sellers_dataset.csv') # Correcto

Ejercicio 3.1. Número de pedidos que llegan tarde por ciudad

In [3]:
df_orders_all['order_delivered_customer_date'] = pd.to_datetime(df_orders_all['order_delivered_customer_date'])
df_orders_all['order_estimated_delivery_date'] = pd.to_datetime(df_orders_all['order_estimated_delivery_date'])

days_measurement = (df_orders_all['order_delivered_customer_date'] - df_orders_all['order_estimated_delivery_date']).dt.days

df_late_orders = df_orders_all[
    (df_orders_all['order_delivered_customer_date'] > df_orders_all['order_estimated_delivery_date']) 
    & (df_orders_all['order_status'] == 'delivered') & (days_measurement > 0)
]

df_late_orders_city = pd.merge(df_late_orders, df_customers_all, on='customer_id')

df_late_orders_city['state_city'] = (df_late_orders_city['customer_city'].str.capitalize() + ' (' + df_late_orders_city['customer_state'] + ')')

df_late_orders_city

df_late_orders_city.groupby('state_city').size().sort_values(ascending=False).to_frame().reset_index().rename(columns={'customer_city' : 'Ciudad', 0 : 'Cant. Pedidos'}).head(n=25)

,state_city,Cant. Pedidos
0,Sao paulo (SP),715
1,Rio de janeiro (RJ),706
2,Salvador (BA),174
3,Belo horizonte (MG),137
4,Porto alegre (RS),136
5,Campinas (SP),119
6,Brasilia (DF),118
7,Niteroi (RJ),96
8,Fortaleza (CE),94
9,Sao goncalo (RJ),83


Ejercicio 3.2. Porcentaje de pedidos retrasados respecto al total de pedidos de la ciudad

In [4]:
df_customers_orders_all = pd.merge(df_orders_all, df_customers_all, on='customer_id')

df_customers_orders_all['state_city'] = (df_customers_orders_all['customer_city'].str.capitalize() + ' (' + df_customers_orders_all['customer_state'] + ')')

df_orders_percentage = pd.merge(df_customers_orders_all.groupby('state_city').size().reset_index(name='total_orders'),
                                df_late_orders_city.groupby('state_city').size().reset_index(name='total_late_orders'), on='state_city', how='left').fillna(0)

df_orders_percentage['percentage'] = round((df_orders_percentage['total_late_orders'] / 
                                      df_orders_percentage['total_orders']) * 100, 2)

df_orders_percentage.sort_values(by=['total_late_orders'], ascending=[False]).reset_index().rename(columns={'state_city' : 'Ciudad', 'percentage' : 'Porcentaje'}).head(n=25)

,index,Ciudad,total_orders,total_late_orders,Porcentaje
0,3758,Sao paulo (SP),15540,715.0,4.60
1,3281,Rio de janeiro (RJ),6882,706.0,10.26
2,3375,Salvador (BA),1245,174.0,13.98
3,469,Belo horizonte (MG),2773,137.0,4.94
4,3080,Porto alegre (RS),1379,136.0,9.86
5,739,Campinas (SP),1444,119.0,8.24
6,587,Brasilia (DF),2131,118.0,5.54
7,2553,Niteroi (RJ),849,96.0,11.31
8,1426,Fortaleza (CE),654,94.0,14.37
9,3625,Sao goncalo (RJ),409,83.0,20.29


Ejercicio 3.3 Tiempo medio de retraso en días

In [5]:
df_mean_time_days = df_late_orders_city.copy()
df_mean_time_days['order_delivered_customer_date'] = df_mean_time_days['order_delivered_customer_date'].astype('date64[pyarrow]')
df_mean_time_days['order_estimated_delivery_date'] = df_mean_time_days['order_estimated_delivery_date'].astype('date64[pyarrow]')

df_mean_time_days['late_days'] = (df_mean_time_days['order_delivered_customer_date'] - df_mean_time_days['order_estimated_delivery_date']).dt.days

df_mean_time_days.groupby('state_city')['late_days'].mean().sort_values(ascending=False).to_frame().rename(columns={'state_city' : 'Ciudad (Estado)', 'late_days' : 'Media Dias de Retraso'})

,Media Dias de Retraso
state_city,
Montanha (ES),181.0
Perdizes (MG),162.0
Teutonia (RS),153.0
Formosa (GO),152.0
Macapa (AP),144.0
...,...
Sidrolandia (MS),1.0
Teotonio vilela (AL),1.0
Tururu (CE),1.0


Determinar causa de insatisfacción de los clientes

In [6]:
df_late_orders_reviews = pd.merge(df_mean_time_days, df_order_reviews_all, on='order_id', how='left')

bins = [0,2,5,10,20, int(df_late_orders_reviews['late_days'].max())]
labels = ['0-2 dias', '2-5 dias', '5-10 dias', '10-20 dias', '20+ dias']

df_late_orders_reviews['range'] = pd.cut(df_late_orders_reviews['late_days'], bins=bins, labels=labels)

df_late_orders_rating_ranges = round(df_late_orders_reviews.groupby('range', observed=True)['review_score'].mean(), 2).reset_index()

df_late_orders_count_ranges = df_late_orders_reviews.groupby('range', observed=True).size().reset_index().rename(columns={ 0 : 'count'})

print(df_late_orders_rating_ranges)
print(df_late_orders_count_ranges)

        range  review_score
0    0-2 dias          3.51
1    2-5 dias          2.47
2   5-10 dias          1.77
3  10-20 dias          1.68
4    20+ dias          1.77
        range  count
0    0-2 dias   1374
1    2-5 dias   1403
2   5-10 dias   1680
3  10-20 dias   1305
4    20+ dias    800


Ejercicio 4.1. Número de reviews por estado

In [84]:
raw_df_reviews_state = pd.merge(df_orders_all, df_order_reviews_all, on='order_id')

raw_df_reviews_state['order_delivered_customer_date'] = pd.to_datetime(raw_df_reviews_state['order_delivered_customer_date'])
raw_df_reviews_state['order_estimated_delivery_date'] = pd.to_datetime(raw_df_reviews_state['order_estimated_delivery_date'])

df_reviews_state = raw_df_reviews_state[(raw_df_reviews_state['order_delivered_customer_date'] <= raw_df_reviews_state['order_estimated_delivery_date'])]

raw_df_reviews_state_customer = pd.merge(df_reviews_state, df_customers_all, on='customer_id')

raw_df_reviews_state_customer['state_city'] = (raw_df_reviews_state_customer['customer_city'].str.capitalize() + ' (' + raw_df_reviews_state_customer['customer_state'] + ')')

df_reviews_state_customer = raw_df_reviews_state_customer.groupby(['state_city', 'order_status']).size().to_frame().reset_index().rename(columns={'state_city' : 'Ciudad (Estado)', 'order_status' : 'Estado', 0 : 'Cant. Pedidos'}).sort_values(by='Cant. Pedidos', ascending=False)

df_reviews_state_customer[df_reviews_state_customer['Estado'] == 'delivered'].reset_index().head(n=25)

,index,Ciudad (Estado),Estado,Cant. Pedidos
0,3602,Sao paulo (SP),delivered,14111
1,3144,Rio de janeiro (RJ),delivered,5803
2,451,Belo horizonte (MG),delivered,2537
3,565,Brasilia (DF),delivered,1931
4,1151,Curitiba (PR),delivered,1413
5,712,Campinas (SP),delivered,1262
6,2950,Porto alegre (RS),delivered,1187
7,1538,Guarulhos (SP),delivered,1068
8,3236,Salvador (BA),delivered,979
9,3429,Sao bernardo do campo (SP),delivered,859


Ejercicio 4.2. Score medio de las reviews en cada estado

In [108]:
raw_df_reviews_state_customer_orders = raw_df_reviews_state_customer.groupby(['state_city', 'order_status']).size().to_frame()
raw_df_reviews_state_customer_ratings_mean = raw_df_reviews_state_customer.groupby(['state_city', 'order_status'])['review_score'].mean().to_frame()

raw_df_reviews_state_customer_ratings_orders = pd.merge(raw_df_reviews_state_customer_orders, raw_df_reviews_state_customer_ratings_mean, on=['state_city','order_status'] )
df_reviews_state_customer_ratings_orders = raw_df_reviews_state_customer_ratings_orders.reset_index().sort_values(0, ascending=False).rename(columns={'state_city' : 'Ciudad (Estado)', 'order_status' : 'Estado', 0 : 'Cant. Pedidos', 'review_score' : 'Puntuacion'})
df_reviews_state_customer_ratings_orders['Puntuacion'] = round(df_reviews_state_customer_ratings_orders['Puntuacion'], 2)
df_reviews_state_customer_ratings_orders[df_reviews_state_customer_ratings_orders['Estado'] == 'canceled']

,Ciudad (Estado),Estado,Cant. Pedidos,Puntuacion
3143,Rio de janeiro (RJ),canceled,3,1.0
1365,Florianopolis (SC),canceled,1,1.0
3601,Sao paulo (SP),canceled,1,5.0


Métrica Personalizada 1. Distribución de volumen de ventas según categoría de producto, sobre el total de productos

In [189]:
raw_df_order_items_products = pd.merge(df_order_items_all, df_products_all, on='product_id') 
raw_df_order_items_products = raw_df_order_items_products[raw_df_order_items_products['product_category_name'].notna()]

raw_df_order_items_products = raw_df_order_items_products.groupby('product_category_name').size().sort_values(ascending=False)

raw_df_order_items_products = raw_df_order_items_products.to_frame().reset_index().rename(columns={ 'product_category_name' : 'Categoria', 0 : 'Porcentaje'})

raw_df_order_items_products['Categoria'] = raw_df_order_items_products['Categoria'].str.replace('_', ' ').str.capitalize()
raw_df_order_items_products['Porcentaje'] = (raw_df_order_items_products['Porcentaje'] / raw_df_order_items_products['Porcentaje'].sum()) * 100

raw_df_order_items_products


,Categoria,Porcentaje
0,Cama mesa banho,10.009275
1,Beleza saude,8.708025
2,Esporte lazer,7.781390
3,Moveis decoracao,7.504930
4,Informatica acessorios,7.048367
...,...,...
68,Cds dvds musicais,0.012607
69,La cuisine,0.012607
70,Pc gamer,0.008105
71,Fashion roupa infanto juvenil,0.007204
